We now have our Elo table in place. We seek to find the relationship via ML between Elo diff, and expected number of goals. Currently, our Elo calculation produces a final Elo dictionary for each team, so we need to alter the logic to be able to retrieve the Elo before each match.

In [3]:
import sqlite3
import pandas as pd
import soccerdata as sd

connection = sqlite3.connect("../data/processed/football.db")

matches = pd.read_sql_query(
    "SELECT * FROM matches",
    connection
)

matches["date"] = pd.to_datetime(matches["date"])
matches = matches.sort_values("date").reset_index(drop=True)

In [4]:
clubelo = sd.ClubElo()

elo_2015 = clubelo.read_by_date("2015-06-30")

countries = ["ENG", "SCO", "GER", "ESP", "ITA", "FRA", "BEL", "NED", "POR", "TUR", "GRE"]

top_leagues = elo_2015[
    (elo_2015["country"].isin(countries)) &
    (elo_2015["level"] == 1)
]

league_elos = top_leagues.groupby("country")["elo"].mean()

league_to_country = {
    "E0": "ENG",
    "SC0": "SCO",
    "D1": "GER",
    "SP1": "ESP",
    "I1": "ITA",
    "F1": "FRA",
    "B1": "BEL",
    "N1": "NED",
    "P1": "POR",
    "T1": "TUR",
    "G1": "GRE"
}

team_leagues = pd.read_sql_query(
    "SELECT DISTINCT team_id, league FROM team_aliases WHERE source = 'football_data'",
    connection
)

[08/13/26 16:00:58] INFO     Saving cached data to C:\Users\rohan\soccerdata\data\ClubElo            ]8;id=2644347;file://c:\Users\rohan\Documents\UCL-MU-Predictor\.venv\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=2644348;file://c:\Users\rohan\Documents\UCL-MU-Predictor\.venv\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

[2026-08-13 16:00:58] INFO     TLSLibrary:_load_library:397 - Successfully loaded TLS library: C:\Users\rohan\Documents\UCL-MU-Predictor\.venv\Lib\site-packages\tls_requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll


                    INFO     Successfully loaded TLS library:                                      ]8;id=2644355;file://c:\Users\rohan\Documents\UCL-MU-Predictor\.venv\Lib\site-packages\tls_requests\models\libraries.py\libraries.py]8;;\:]8;id=2644356;file://c:\Users\rohan\Documents\UCL-MU-Predictor\.venv\Lib\site-packages\tls_requests\models\libraries.py#397\397]8;;\
                             C:\Users\rohan\Documents\UCL-MU-Predictor\.venv\Lib\site-packages\tls                 
                             _requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll                                 

In [ ]:
# building auxilliary table (key match_id) to matches, with per match elo data (elo before, elo after etc.)
# join to full table, to give full match data with elo 

elo = {}

for _, team in team_leagues.iterrows():

    league = team["league"]
    country = league_to_country[league]

    elo[team["team_id"]] = league_elos.loc[country]


home_elo_before = []
away_elo_before = []

home_elo_after = []
away_elo_after = []

expected_home_scores = []
expected_away_scores = []

K = 20

for _, match in matches.iterrows():

    if match["home_team_id"] not in elo:
        elo[match["home_team_id"]] = 1500

    if match["away_team_id"] not in elo:
        elo[match["away_team_id"]] = 1500

    current_home_rating = elo[match["home_team_id"]]
    current_away_rating = elo[match["away_team_id"]]

    home_elo_before.append(current_home_rating)
    away_elo_before.append(current_away_rating)

    Q_A = 10 ** (current_home_rating / 400)
    Q_B = 10 ** (current_away_rating / 400)

    expected_home_score = Q_A / (Q_A + Q_B)
    expected_away_score = Q_B / (Q_A + Q_B)

    expected_home_scores.append(expected_home_score)
    expected_away_scores.append(expected_away_score)

    if match["home_goals"] > match["away_goals"]:
        S_A = 1
        S_B = 0

    elif match["home_goals"] == match["away_goals"]:
        S_A = 0.5
        S_B = 0.5

    else:
        S_A = 0
        S_B = 1

    updated_home_rating = current_home_rating + K * (S_A - expected_home_score)
    updated_away_rating = current_away_rating + K * (S_B - expected_away_score)

    elo[match["home_team_id"]] = updated_home_rating
    elo[match["away_team_id"]] = updated_away_rating

    home_elo_after.append(updated_home_rating)
    away_elo_after.append(updated_away_rating)


elo_matches = matches.copy()

elo_matches["home_elo_before"] = home_elo_before
elo_matches["away_elo_before"] = away_elo_before

elo_matches["home_elo_after"] = home_elo_after
elo_matches["away_elo_after"] = away_elo_after

elo_matches["expected_home_score"] = expected_home_scores
elo_matches["expected_away_score"] = expected_away_scores

elo_matches["elo_diff"] = (
    elo_matches["home_elo_before"] - elo_matches["away_elo_before"]
)

elo_matches.head()

,match_id,date,season,competition,home_team_id,away_team_id,home_team_name,away_team_name,home_goals,away_goals,source,home_elo_before,away_elo_before,home_elo_after,away_elo_after,expected_home_score,expected_away_score,elo_diff
0,1,2015-06-30,2015/16,CL,14,384,Pyunik,Folgore,2,1,footystats,1500.0,1500.0,1510.0,1490.0,0.5,0.5,0.0
1,2,2015-06-30,2015/16,CL,184,8,Lincoln Red Imps,FC Santa Coloma,0,0,footystats,1500.0,1500.0,1500.0,1500.0,0.5,0.5,0.0
2,3,2015-06-30,2015/16,CL,326,104,Crusaders,Levadia Tallinn,0,0,footystats,1500.0,1500.0,1500.0,1500.0,0.5,0.5,0.0
3,4,2015-07-01,2015/16,CL,106,503,B36 Torshavn,The New Saints,1,2,footystats,1500.0,1500.0,1490.0,1510.0,0.5,0.5,0.0
4,5,2015-07-07,2015/16,CL,8,184,FC Santa Coloma,Lincoln Red Imps,1,2,footystats,1500.0,1500.0,1490.0,1510.0,0.5,0.5,0.0
